# Notebook 03: Seed-count ratio as an a priori predictor

Section 4.3.4 of the paper observes that the EM-to-baseline
seed-count ratio (a quantity available before the baseline
condition is even run) is monotonically aligned with the
post-hoc exclusive-bug proportion p_excl across the three
campaigns. This notebook checks the ordering and visualises it.

The check is a direction-of-relationship statement, not a
numeric threshold. With n = 3 campaigns, only the ordering can
be sanity-checked; nothing can be said about a quantitative
predictor. Confounding through target structure cannot be ruled
out at this scale and is discussed in the paper.


In [ ]:
import csv
from pathlib import Path

import matplotlib.pyplot as plt

DATA_DIR = Path("..") / "data"
with (DATA_DIR / "rq2_discovery.csv").open(newline="") as handle:
    discovery = list(csv.DictReader(handle))


## Compute seed ratios


In [ ]:
rows = []
for row in discovery:
    seeds_em = int(row["seeds_em"])
    seeds_base = int(row["seeds_base"])
    p_excl = float(row["p_excl"])
    seed_ratio = seeds_em / seeds_base
    rows.append({
        "campaign": row["campaign"],
        "seeds_em": seeds_em,
        "seeds_base": seeds_base,
        "seed_ratio": seed_ratio,
        "p_excl": p_excl,
    })

print(f"{'Campaign':<10}{'Seeds(EM)':>11}{'Seeds(B)':>10}{'Ratio':>10}{'p_excl':>10}")
print("-" * 51)
for r in rows:
    print(f"{r['campaign']:<10}{r['seeds_em']:>11}{r['seeds_base']:>10}"
          f"{r['seed_ratio']:>10.2f}{r['p_excl']:>10.2f}")


Expected seed ratios: C2 = 14.0, C3 = 1.09, picoc = 97.0.


## Verify ordering match


In [ ]:
by_seed_ratio = sorted(rows, key=lambda r: r["seed_ratio"])
by_p_excl = sorted(rows, key=lambda r: r["p_excl"])

ord_seed = [r["campaign"] for r in by_seed_ratio]
ord_pexcl = [r["campaign"] for r in by_p_excl]

print(f"Order by seed ratio (asc):  {ord_seed}")
print(f"Order by p_excl (asc):      {ord_pexcl}")
match = all(a == b for a, b in zip(ord_seed, ord_pexcl))
print(f"Orderings match position by position: {match}")
assert ord_seed == ["C3", "C2", "picoc"], ord_seed
assert ord_pexcl == ["C3", "C2", "picoc"], ord_pexcl


## Visualisation


In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
for r in rows:
    ax.scatter(r["seed_ratio"], r["p_excl"], s=80, color="#1f4e79")
    ax.annotate(r["campaign"],
                (r["seed_ratio"], r["p_excl"]),
                xytext=(8, 6), textcoords="offset points")
ax.set_xscale("log")
ax.set_xlabel("Seed ratio EM / baseline (log scale)")
ax.set_ylabel("p_excl (exclusive-bug proportion)")
ax.set_title("A priori seed ratio vs. post-hoc exclusive-bug proportion")
ax.grid(True, which="both", linestyle=":", alpha=0.5)
fig.tight_layout()
plt.show()


## Caveats

n = 3 campaigns is the absolute minimum at which a monotonic
ordering check is meaningful. The result supports the direction
of the relationship between EM-anomalous seed abundance and the
share of bugs found exclusively from the EM-anomalous regime.
It does not establish a robust statistical claim, and it does
not propose a numeric threshold for the ratio. Confounding
through target structure (instruction mix, allocation patterns,
I/O rate) cannot be ruled out at this scale.
